# Depression Detection – Text Pipeline (BERT)
**Dataset:** DAIC-WOZ | **Model:** bert-base-uncased | **Validation:** 10-fold CV

Authors: Ayush Negi (22BLC1259), Ayush Arora (22BLC1218), Karan Ghuwalewala (22BLC1201)

VIT Chennai – School of Electronics Engineering, Nov 2025

In [ ]:
# Mount Google Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install transformers torch scikit-learn pandas numpy -q

## 1. Imports & Config

In [ ]:
import os, glob, numpy as np, pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.utils import resample

MODEL_NAME     = 'bert-base-uncased'
TRANSCRIPT_DIR = '/content/drive/MyDrive/DDS/data/*.csv'
LABELS_CSV     = '/content/drive/MyDrive/DDS/data/labels.csv'
MAX_LEN, BATCH_SIZE, EPOCHS = 128, 8, 10
LEARNING_RATE, WEIGHT_DECAY = 2e-5, 0.01
N_SPLITS, SEED = 10, 42

INTERROGATIVE_WORDS = (
    'what','when','where','which','who','whom','whose','why','how',
    'is','are','am','was','were','do','does','did','has','have','had',
    'can','could','will','would','should','may','might',
    'tell me','describe','explain','give me an example',
    'so,','and what','what about','how about'
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Extract Q&A Pairs from Transcripts

In [ ]:
all_qa = []
for file_path in glob.glob(TRANSCRIPT_DIR):
    person_id = os.path.basename(file_path).split('_')[0]
    try:
        df = pd.read_csv(file_path, sep='\t', names=['Speaker','value'], header=0)
        current_q, current_a = '', []
        for _, row in df.iterrows():
            speaker  = str(row['Speaker']).strip()
            dialogue = str(row['value']).strip()
            if speaker == 'Ellie':
                if current_q and current_a:
                    all_qa.append({'personId':person_id,'question':current_q,'answer':' '.join(current_a)})
                current_q, current_a = dialogue, []
            elif speaker == 'Participant' and current_q and dialogue:
                current_a.append(dialogue)
        if current_q and current_a:
            all_qa.append({'personId':person_id,'question':current_q,'answer':' '.join(current_a)})
    except Exception as e:
        print(f'Error: {file_path} – {e}')

df_qa = pd.DataFrame(all_qa)
df_qa.dropna(subset=['question','answer'], inplace=True)
df_qa = df_qa[df_qa['question'].str.strip() != '']
df_qa = df_qa[df_qa['answer'].str.strip() != '']
df_qa = df_qa[df_qa['question'].str.lower().str.startswith(INTERROGATIVE_WORDS)]
print(f'Total Q&A pairs: {len(df_qa)}')
df_qa.head()

## 3. Merge Labels & Balance Dataset

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)
labels_df['Participant_ID'] = labels_df['Participant_ID'].astype(int)
df_qa['personId'] = df_qa['personId'].astype(int)
merged = df_qa.merge(labels_df, left_on='personId', right_on='Participant_ID', how='inner')
merged.drop(columns=['Participant_ID'], inplace=True)

majority  = merged[merged['PHQ_Binary'] == 0]
minority  = merged[merged['PHQ_Binary'] == 1]
print(f'Before: Not-Depressed={len(majority)}, Depressed={len(minority)}')

minority_up = resample(minority, replace=True, n_samples=len(majority), random_state=SEED)
df_balanced = pd.concat([majority, minority_up]).sample(frac=1, random_state=SEED).reset_index(drop=True)
df_balanced['text'] = df_balanced['question'] + ' [SEP] ' + df_balanced['answer']
print(f'After balancing: {len(df_balanced)} samples')
print(df_balanced['PHQ_Binary'].value_counts())

## 4. PyTorch Dataset & DataLoader

In [ ]:
class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer.encode_plus(
            self.texts[idx], add_special_tokens=True,
            max_length=self.max_len, return_token_type_ids=False,
            padding='max_length', truncation=True,
            return_attention_mask=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)}

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded.')

## 5. Train & Evaluate Functions

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train(); losses = []
    for d in loader:
        ids  = d['input_ids'].to(device)
        mask = d['attention_mask'].to(device)
        lbls = d['labels'].to(device)
        out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
        loss = out.loss; losses.append(loss.item())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
    return np.mean(losses)

def eval_model(model, loader):
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for d in loader:
            ids  = d['input_ids'].to(device)
            mask = d['attention_mask'].to(device)
            lbls = d['labels'].to(device)
            out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
            probs = torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs); all_labels.extend(lbls.cpu().numpy())
    preds = (np.array(all_probs) >= 0.5).astype(int)
    return {'accuracy': accuracy_score(all_labels, preds),
            'f1':       f1_score(all_labels, preds),
            'roc_auc':  roc_auc_score(all_labels, all_probs),
            'labels': all_labels, 'preds': preds}

## 6. 10-Fold Cross-Validation

In [ ]:
kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_f1, fold_roc, fold_acc = [], [], []
master_labels, master_preds = [], []

for fold, (train_ids, test_ids) in enumerate(kfold.split(df_balanced)):
    print(f'\n──── Fold {fold+1}/{N_SPLITS} ────')
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    df_tr = df_balanced.iloc[train_ids]; df_te = df_balanced.iloc[test_ids]

    tr_ds = DepressionDataset(df_tr['text'].to_numpy(), df_tr['PHQ_Binary'].to_numpy(), tokenizer, MAX_LEN)
    te_ds = DepressionDataset(df_te['text'].to_numpy(), df_te['PHQ_Binary'].to_numpy(), tokenizer, MAX_LEN)
    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
    te_dl = DataLoader(te_ds, batch_size=BATCH_SIZE)

    optimizer   = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=len(tr_dl)*EPOCHS)

    for epoch in range(EPOCHS):
        loss = train_epoch(model, tr_dl, optimizer, scheduler)
        print(f'  Epoch {epoch+1}/{EPOCHS}  Loss: {loss:.4f}')

    m = eval_model(model, te_dl)
    print(f'  F1={m["f1"]:.4f}  ROC-AUC={m["roc_auc"]:.4f}  Acc={m["accuracy"]:.4f}')
    fold_f1.append(m['f1']); fold_roc.append(m['roc_auc']); fold_acc.append(m['accuracy'])
    master_labels.extend(m['labels']); master_preds.extend(m['preds'])

print(f'\nAverage F1:  {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}')
print(f'Average ROC: {np.mean(fold_roc):.4f} ± {np.std(fold_roc):.4f}')
print(f'Average Acc: {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}')

## 7. Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
cm = confusion_matrix(master_labels, master_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Depression','Depression'],
            yticklabels=['No Depression','Depression'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Aggregate Confusion Matrix (All Folds)')
plt.tight_layout(); plt.show()